In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)
df.head()

## **EDA**

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
# Visualizing missing values
plt.figure(figsize=(12, 6))

sns.heatmap(df.isnull(), cbar=False, cmap='viridis')

plt.title("Missing Values Heatmap")
plt.show()

In [ ]:
df.info()

In [ ]:
# Compute mean age before imputation, then fill missing values
mean_age = df["Age"].mean()
df["Age"] = df["Age"].fillna(mean_age)

In [ ]:
df["Age"].mean()

In [ ]:
df["Age"] = df["Age"].round().astype("int64")

In [ ]:
df.info()

In [ ]:
df.columns.tolist()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df[df.duplicated()]

In [ ]:
# Separating numerical and categorical columns
# Numerical Columns
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
numerical_cols

In [ ]:
# Categorical Columns
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()
categorical_cols

In [ ]:
# Survival percentage by passenger class
plt.figure(figsize=(8, 5))
sns.barplot(data=df, x='Pclass', y='Survived')

plt.title("Survival Rate by Passenger Class")
plt.ylabel("Survival Rate")
plt.show()

In [ ]:
# Survival percentage by gender
plt.figure(figsize=(8, 5))
sns.barplot(data=df, x='Sex', y='Survived', order=['female', 'male'])

plt.title("Survival Rate by Gender")
plt.xticks([0, 1], ['Female', 'Male'])
plt.ylabel("Survival Rate")
plt.show()

In [ ]:
# Fare vs Survival
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='Survived', y='Fare')

plt.title("Fare Distribution by Survival")
plt.show()

## **Feature Engineering**

In [ ]:
# Selecting only required columns
df = df[['Pclass', 'Age', 'Sex', 'Fare', 'Survived']].copy()
df.head()

In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
le = LabelEncoder()
df["Sex"] = le.fit_transform(df["Sex"])
print(dict(zip(le.classes_, le.transform(le.classes_))))

In [ ]:
df.head()

In [ ]:
plt.figure(figsize=(12, 8))

for i, col in enumerate(df.columns, 1):
    plt.subplot(2, 3, i)
    sns.histplot(df[col], kde=True, bins=20)
    plt.title(f"Distribution of {col}")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')

plt.title("Feature Correlation Heatmap")
plt.show()

## **Separating Independent and Dependent Features**

In [ ]:
X = df[["Pclass", "Age", "Sex", "Fare"]]
y = df["Survived"]

In [ ]:
X.head()

In [ ]:
X.shape

In [ ]:
y.head()

In [ ]:
y.shape

## **Train-Test Split & Feature Scaling**

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train = X_train.copy()
X_test = X_test.copy()
X_train_raw = X_train.copy()  # Kept unscaled for leak-free cross-validation

In [ ]:
X_train.shape

In [ ]:
y_train.shape

In [ ]:
X_test.shape

In [ ]:
fare_original = X_train["Fare"].copy()
fare_original.head()

In [ ]:
from sklearn.preprocessing import StandardScaler

# Fit scaler only on training data to prevent data leakage
scaler = StandardScaler()
X_train["Fare"] = scaler.fit_transform(X_train[["Fare"]])
X_test["Fare"] = scaler.transform(X_test[["Fare"]])

In [ ]:
X_train.head()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

# Before standardization
sns.histplot(fare_original, kde=True, ax=ax[0])
ax[0].set_title("Fare Before Standardization (Train)")

# After standardization
sns.histplot(X_train['Fare'], kde=True, ax=ax[1])
ax[1].set_title("Fare After Standardization (Train)")

plt.tight_layout()
plt.show()

## **Training & Evaluation**

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(random_state=42)
lr.fit(X_train, y_train)

In [ ]:
y_pred = lr.predict(X_test)
y_pred

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues')

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

y_prob = lr.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(8, 5))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, y_prob):.3f}")
plt.plot([0, 1], [0, 1], linestyle='--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

| AUC Score | Model Performance |
| :--- | :--- |
| 1.0 | Perfect |
| 0.9–1.0 | Excellent |
| 0.8–0.9 | Good |
| 0.7–0.8 | Fair |
| 0.6–0.7 | Poor |
| 0.5 | Random Guessing |

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Use a Pipeline so StandardScaler is fit inside each CV fold without data leakage
cv_pipeline = Pipeline([
    ("scaler", ColumnTransformer([("fare", StandardScaler(), ["Fare"])], remainder="passthrough")),
    ("lr", LogisticRegression(random_state=42))
])

cv_scores = cross_val_score(cv_pipeline, X_train_raw, y_train, cv=5, scoring='accuracy')

print("CV Scores:", cv_scores)
print("Mean CV Accuracy:", cv_scores.mean())
print("Standard Deviation:", cv_scores.std())

In [ ]:
new_passengers = pd.DataFrame({
    'Pclass': [1, 3, 2],
    'Age': [25, 35, 10],
    'Sex': ['female', 'male', 'female'],
    'Fare': [100, 7.25, 30]
})

# Encode gender
new_passengers['Sex'] = new_passengers['Sex'].map({'female': 0, 'male': 1})

# Apply previously fitted scaler
new_passengers['Fare'] = scaler.transform(new_passengers[['Fare']])

# Predict
predictions = lr.predict(new_passengers)
print(predictions)